# Delta Lake — Funcionalidades Avanzadas

## Unidad 4: Infraestructura de Datos

En el notebook de Lakehouse usamos Delta Lake para crear tablas, hacer appends, time travel y merge basico. Este notebook profundiza en las funcionalidades avanzadas: el transaction log por dentro, merge condicional, particionamiento Delta, optimizacion, vacuum y consultas con DuckDB y Polars.

### Contenido:
1. El transaction log por dentro
2. Merge avanzado (condicional, delete, update selectivo)
3. Tablas Delta particionadas
4. Optimizacion: compactacion y vacuum
5. Consultas con DuckDB (SQL sin Spark)
6. Consultas con Polars
7. Pipeline incremental

In [ ]:
# !pip install deltalake

import pandas as pd
import numpy as np
import os
import shutil
import json
from datetime import datetime, timedelta

try:
    from deltalake import DeltaTable, write_deltalake
    DELTA_OK = True
    print("deltalake OK")
except ImportError:
    DELTA_OK = False
    print("Instala con: pip install deltalake")

In [ ]:
# ============================================================
# CREAR UNA TABLA DE EJEMPLO
# ============================================================

TABLE = 'delta_avanzado/ventas'
if os.path.exists('delta_avanzado'):
    shutil.rmtree('delta_avanzado')

np.random.seed(42)

df_inicial = pd.DataFrame({
    'venta_id': range(1, 201),
    'fecha': np.random.choice(pd.date_range('2024-06-01', '2024-06-30'), 200),
    'producto': np.random.choice(['dashboard', 'reporte', 'api', 'app_web', 'pipeline'], 200),
    'region': np.random.choice(['bogota', 'medellin', 'cali', 'manizales'], 200, p=[0.4, 0.25, 0.2, 0.15]),
    'unidades': np.random.randint(1, 30, 200),
    'precio': np.round(np.random.uniform(200, 800, 200), 2),
    'estado': np.random.choice(['completada', 'pendiente', 'cancelada'], 200, p=[0.7, 0.2, 0.1]),
})
df_inicial['ingreso'] = df_inicial['unidades'] * df_inicial['precio']

if DELTA_OK:
    write_deltalake(TABLE, df_inicial, mode='overwrite')
    print(f"Tabla creada: {len(df_inicial)} filas")
    print(f"Columnas: {df_inicial.columns.tolist()}")
    print(f"Estados: {df_inicial['estado'].value_counts().to_dict()}")

---
## 1. El transaction log por dentro

In [ ]:
# ============================================================
# LEER EL CONTENIDO DEL _delta_log
# ============================================================

if DELTA_OK:
    log_dir = f'{TABLE}/_delta_log'
    
    print("Archivos en _delta_log/:\n")
    for f in sorted(os.listdir(log_dir)):
        size = os.path.getsize(os.path.join(log_dir, f))
        print(f"  {f:50s} {size:>6,} bytes")
    
    # Leer el primer commit
    primer_commit = os.path.join(log_dir, '00000000000000000000.json')
    print(f"\nContenido del primer commit (version 0):\n")
    
    with open(primer_commit) as f:
        for i, linea in enumerate(f):
            entry = json.loads(linea)
            # Cada linea es una accion: commitInfo, metaData, protocol, add
            tipo = list(entry.keys())[0]
            if tipo == 'add':
                print(f"  [{tipo}] archivo={entry['add']['path'][:40]}...")
                print(f"         size={entry['add']['size']:,} bytes")
                stats = json.loads(entry['add'].get('stats', '{}'))
                if stats:
                    print(f"         filas={stats.get('numRecords', '?')}")
            elif tipo == 'commitInfo':
                print(f"  [{tipo}] operation={entry['commitInfo'].get('operation')}")
            elif tipo == 'metaData':
                schema = json.loads(entry['metaData'].get('schemaString', '{}'))
                cols = [c['name'] for c in schema.get('fields', [])]
                print(f"  [{tipo}] columnas={cols}")
            elif tipo == 'protocol':
                print(f"  [{tipo}] minReader={entry['protocol'].get('minReaderVersion')}, "
                      f"minWriter={entry['protocol'].get('minWriterVersion')}")
            else:
                print(f"  [{tipo}]")

In [ ]:
# ============================================================
# HACER OPERACIONES Y VER COMO CAMBIA EL LOG
# ============================================================

if DELTA_OK:
    # Operacion 1: append de 10 filas
    df_append = pd.DataFrame({
        'venta_id': range(201, 211),
        'fecha': pd.Timestamp('2024-07-01'),
        'producto': ['dashboard'] * 10,
        'region': ['bogota'] * 10,
        'unidades': np.random.randint(5, 20, 10),
        'precio': [450.0] * 10,
        'estado': ['completada'] * 10,
    })
    df_append['ingreso'] = df_append['unidades'] * df_append['precio']
    write_deltalake(TABLE, df_append, mode='append')
    
    # Ver el nuevo commit
    print("Despues del append:\n")
    commit_1 = os.path.join(log_dir, '00000000000000000001.json')
    with open(commit_1) as f:
        for linea in f:
            entry = json.loads(linea)
            tipo = list(entry.keys())[0]
            if tipo == 'add':
                stats = json.loads(entry['add'].get('stats', '{}'))
                print(f"  [{tipo}] archivo nuevo con {stats.get('numRecords', '?')} filas")
            elif tipo == 'commitInfo':
                print(f"  [{tipo}] {entry['commitInfo'].get('operation')}")
    
    print(f"\nEl append solo agrega un nuevo archivo Parquet.")
    print(f"No toca los archivos anteriores.")

In [ ]:
# ============================================================
# ESQUEMA DE LA TABLA
# ============================================================

if DELTA_OK:
    dt = DeltaTable(TABLE)
    schema = dt.schema()
    
    print("Esquema de la tabla Delta:\n")
    for field in schema.to_pyarrow():
        nullable = 'nullable' if field.nullable else 'not null'
        print(f"  {field.name:15s} {str(field.type):15s} {nullable}")
    
    # Archivos fisicos que componen la tabla
    files = dt.file_uris()
    print(f"\nArchivos Parquet activos: {len(files)}")
    for f in files:
        print(f"  {os.path.basename(f)}")

---
## 2. Merge avanzado

In [ ]:
# ============================================================
# MERGE CONDICIONAL: actualizar solo ciertos campos
# ============================================================

# Escenario: llegan correcciones de precio para algunas ventas
# Solo queremos actualizar el precio e ingreso, no tocar lo demas

correcciones = pd.DataFrame({
    'venta_id': [5, 15, 25, 999],  # 999 no existe → se ignora
    'precio_corregido': [550.0, 320.0, 710.0, 100.0],
})

if DELTA_OK:
    dt = DeltaTable(TABLE)
    
    # Antes
    df_antes = dt.to_pandas()
    print("Antes de la correccion:")
    print(df_antes[df_antes['venta_id'].isin([5, 15, 25])][['venta_id', 'precio', 'ingreso']].to_string(index=False))
    
    # Merge con update selectivo
    (
        dt.merge(
            source=correcciones,
            predicate='s.venta_id = t.venta_id',
            source_alias='s',
            target_alias='t',
        )
        .when_matched_update({  # Solo actualizar estos campos
            'precio': 's.precio_corregido',
            'ingreso': 't.unidades * s.precio_corregido',
        })
        # No hay when_not_matched → venta_id=999 se ignora
        .execute()
    )
    
    # Despues
    dt = DeltaTable(TABLE)
    df_despues = dt.to_pandas()
    print("\nDespues de la correccion:")
    print(df_despues[df_despues['venta_id'].isin([5, 15, 25])][['venta_id', 'precio', 'ingreso']].to_string(index=False))
    print(f"\nventa_id=999 no existia, no se inserto (correcto).")
else:
    print("[ref] when_matched_update({'precio': 's.precio_corregido'})")
    print("Solo actualiza los campos especificados, no todos.")

In [ ]:
# ============================================================
# MERGE CON DELETE: eliminar registros que matchean
# ============================================================

# Escenario: nos avisan que 3 ventas fueron fraudulentas
# Hay que eliminarlas de la tabla

fraudes = pd.DataFrame({
    'venta_id': [10, 20, 30],
})

if DELTA_OK:
    dt = DeltaTable(TABLE)
    antes = len(dt.to_pandas())
    
    (
        dt.merge(
            source=fraudes,
            predicate='s.venta_id = t.venta_id',
            source_alias='s',
            target_alias='t',
        )
        .when_matched_delete()  # Si existe: eliminar
        .execute()
    )
    
    dt = DeltaTable(TABLE)
    despues = len(dt.to_pandas())
    print(f"Antes: {antes} filas")
    print(f"Despues: {despues} filas")
    print(f"Eliminadas: {antes - despues} filas fraudulentas")
    
    # Las filas no se borraron fisicamente — se marcaron como 'remove' en el log
    # Con time travel aun se pueden ver en la version anterior
    print(f"\nPero con time travel, la version anterior aun las tiene.")
else:
    print("[ref] .when_matched_delete() elimina las filas que matchean.")

In [ ]:
# ============================================================
# MERGE COMPLETO: update + insert + delete en una operacion
# ============================================================

# Escenario: llega la actualizacion diaria del ERP
# - Ventas existentes: actualizar estado
# - Ventas nuevas: insertar
# - Ventas canceladas: marcar como cancelada (no borrar)

actualizacion = pd.DataFrame({
    'venta_id': [1, 2, 3, 211, 212],
    'fecha': pd.Timestamp('2024-07-02'),
    'producto': ['dashboard', 'reporte', 'api', 'pipeline', 'app_web'],
    'region': ['bogota', 'medellin', 'cali', 'bogota', 'manizales'],
    'unidades': [10, 5, 15, 20, 8],
    'precio': [450.0, 280.0, 620.0, 500.0, 350.0],
    'estado': ['completada', 'cancelada', 'completada', 'pendiente', 'completada'],
    'ingreso': [4500.0, 1400.0, 9300.0, 10000.0, 2800.0],
})

if DELTA_OK:
    dt = DeltaTable(TABLE)
    antes = len(dt.to_pandas())
    
    (
        dt.merge(
            source=actualizacion,
            predicate='s.venta_id = t.venta_id',
            source_alias='s',
            target_alias='t',
        )
        .when_matched_update_all()      # Existentes: actualizar todo
        .when_not_matched_insert_all()  # Nuevos: insertar
        .execute()
    )
    
    dt = DeltaTable(TABLE)
    despues = len(dt.to_pandas())
    print(f"Merge completo:")
    print(f"  Antes: {antes} filas")
    print(f"  Despues: {despues} filas")
    print(f"  Actualizadas: 3 (ids 1, 2, 3)")
    print(f"  Insertadas: {despues - antes} (ids 211, 212)")

---
## 3. Tablas Delta particionadas

In [ ]:
# ============================================================
# CREAR TABLA PARTICIONADA POR REGION
# ============================================================

TABLE_PART = 'delta_avanzado/ventas_particionada'

if DELTA_OK:
    # Generar mas datos
    np.random.seed(42)
    n = 10_000
    df_part = pd.DataFrame({
        'venta_id': range(1, n + 1),
        'fecha': np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), n),
        'producto': np.random.choice(['dashboard', 'reporte', 'api', 'app_web'], n),
        'region': np.random.choice(['bogota', 'medellin', 'cali', 'manizales'], n, p=[0.4, 0.25, 0.2, 0.15]),
        'unidades': np.random.randint(1, 30, n),
        'precio': np.round(np.random.uniform(200, 800, n), 2),
    })
    df_part['ingreso'] = df_part['unidades'] * df_part['precio']
    
    # Escribir particionado por region
    write_deltalake(
        TABLE_PART, df_part,
        mode='overwrite',
        partition_by=['region']  # Una carpeta por region
    )
    
    # Ver estructura
    print(f"Tabla particionada: {n:,} filas\n")
    print("Estructura de carpetas:")
    for root, dirs, files in os.walk(TABLE_PART):
        level = root.replace(TABLE_PART, '').count(os.sep)
        indent = '  ' * level
        dirname = os.path.basename(root)
        if '_delta_log' in root:
            if level == 1:
                print(f'{indent}_delta_log/ ({len(files)} archivos)')
            continue
        n_parquet = len([f for f in files if f.endswith('.parquet')])
        if n_parquet > 0:
            print(f'{indent}{dirname}/ ({n_parquet} archivos Parquet)')
        elif dirs:
            print(f'{indent}{dirname}/')

In [ ]:
# ============================================================
# LEER SOLO UNA PARTICION
# ============================================================

import time

if DELTA_OK:
    # Leer toda la tabla
    t0 = time.time()
    df_todo = DeltaTable(TABLE_PART).to_pandas()
    t_todo = time.time() - t0
    
    # Leer solo bogota (usando el filtro de particion)
    t0 = time.time()
    dt = DeltaTable(TABLE_PART)
    df_bogota = dt.to_pandas(filters=[('region', '=', 'bogota')])
    t_filtrado = time.time() - t0
    
    print(f"Leer toda la tabla:    {len(df_todo):>6,} filas  {t_todo:.3f} seg")
    print(f"Leer solo bogota:      {len(df_bogota):>6,} filas  {t_filtrado:.3f} seg")
    print(f"\nCon particionamiento, Delta solo lee la carpeta region=bogota/")
    print(f"y salta las demas. Menos datos leidos = mas rapido.")

---
## 4. Optimizacion: compactacion y vacuum

In [ ]:
# ============================================================
# PROBLEMA: muchos archivos pequenos
# ============================================================

# Cada append crea un nuevo archivo Parquet
# Despues de 100 appends, tienes 100+ archivos pequenos
# Eso es lento porque cada archivo tiene overhead de apertura

TABLE_SMALL = 'delta_avanzado/muchos_archivos'

if DELTA_OK:
    # Simular 20 appends pequenos
    for i in range(20):
        df_mini = pd.DataFrame({
            'id': [i * 10 + j for j in range(10)],
            'valor': np.random.rand(10),
        })
        mode = 'overwrite' if i == 0 else 'append'
        write_deltalake(TABLE_SMALL, df_mini, mode=mode)
    
    dt = DeltaTable(TABLE_SMALL)
    n_files = len(dt.file_uris())
    n_rows = len(dt.to_pandas())
    print(f"Despues de 20 appends:")
    print(f"  Filas: {n_rows}")
    print(f"  Archivos Parquet: {n_files}")
    print(f"  Promedio: {n_rows // n_files} filas por archivo")
    print(f"\n  {n_files} archivos de ~10 filas cada uno es ineficiente.")
    print(f"  Cada archivo tiene overhead de apertura/metadata.")

In [ ]:
# ============================================================
# OPTIMIZE: compactar archivos pequenos
# ============================================================

if DELTA_OK:
    dt = DeltaTable(TABLE_SMALL)
    
    # Compactar
    result = dt.optimize.compact()
    
    dt = DeltaTable(TABLE_SMALL)
    n_files_despues = len(dt.file_uris())
    
    print(f"Compactacion:")
    print(f"  Antes: {n_files} archivos")
    print(f"  Despues: {n_files_despues} archivo(s)")
    print(f"  Filas: {len(dt.to_pandas())} (sin cambio)")
    print(f"\n  Mismos datos, menos archivos, consultas mas rapidas.")
    print(f"  Los archivos viejos siguen en disco (para time travel).")
    print(f"  Para borrarlos: vacuum.")

In [ ]:
# ============================================================
# VACUUM: eliminar archivos que ya no se necesitan
# ============================================================

if DELTA_OK:
    dt = DeltaTable(TABLE_SMALL)
    
    # Contar archivos fisicos antes
    archivos_fisicos = [f for f in os.listdir(TABLE_SMALL) if f.endswith('.parquet')]
    archivos_activos = len(dt.file_uris())
    print(f"Archivos fisicos en disco: {len(archivos_fisicos)}")
    print(f"Archivos activos en Delta: {archivos_activos}")
    print(f"Archivos muertos (basura): {len(archivos_fisicos) - archivos_activos}")
    
    # Vacuum: eliminar archivos que no pertenecen a ninguna version reciente
    # retention_hours=0 borra todo lo viejo (en produccion usar 168 = 7 dias)
    try:
        removed = dt.vacuum(
            retention_hours=0,
            enforce_retention_duration=False,  # Permitir 0 horas (solo para demo)
            dry_run=False
        )
        archivos_despues = [f for f in os.listdir(TABLE_SMALL) if f.endswith('.parquet')]
        print(f"\nDespues del vacuum:")
        print(f"  Archivos eliminados: {len(removed)}")
        print(f"  Archivos restantes: {len(archivos_despues)}")
        print(f"\n  CUIDADO: despues del vacuum, no puedes hacer time travel")
        print(f"  a versiones anteriores. Los archivos ya no existen.")
    except Exception as e:
        print(f"  Error: {e}")
        print(f"  En algunas versiones, vacuum requiere Spark.")

---
## 5. Consultas con DuckDB

In [ ]:
# ============================================================
# SQL SOBRE DELTA CON DUCKDB (sin Spark)
# ============================================================

# pip install duckdb

try:
    import duckdb
    DUCK_OK = True
    print(f"DuckDB version: {duckdb.__version__}")
except ImportError:
    DUCK_OK = False
    print("Instala con: pip install duckdb")

In [ ]:
# ============================================================
# CONSULTAS SQL ANALITICAS
# ============================================================

if DELTA_OK and DUCK_OK:
    # Ingreso por region
    print("=== Ingreso por region ===")
    result = duckdb.sql(f"""
        SELECT
            region,
            COUNT(*) AS transacciones,
            SUM(ingreso)::INTEGER AS ingreso_total,
            ROUND(AVG(ingreso), 2) AS ticket_promedio
        FROM delta_scan('{TABLE}')
        GROUP BY region
        ORDER BY ingreso_total DESC
    """).df()
    print(result.to_string(index=False))
    
    # Top 5 productos por unidades
    print("\n=== Top productos por unidades ===")
    result2 = duckdb.sql(f"""
        SELECT
            producto,
            SUM(unidades) AS unidades_total,
            SUM(ingreso)::INTEGER AS ingreso_total
        FROM delta_scan('{TABLE}')
        WHERE estado = 'completada'
        GROUP BY producto
        ORDER BY unidades_total DESC
    """).df()
    print(result2.to_string(index=False))
    
    # Ventas por estado
    print("\n=== Estado de las ventas ===")
    result3 = duckdb.sql(f"""
        SELECT
            estado,
            COUNT(*) AS cantidad,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje
        FROM delta_scan('{TABLE}')
        GROUP BY estado
        ORDER BY cantidad DESC
    """).df()
    print(result3.to_string(index=False))
elif not DUCK_OK:
    print("DuckDB no disponible. Instala con: pip install duckdb")

---
## 6. Consultas con Polars

In [ ]:
# ============================================================
# POLARS: alternativa rapida a pandas con soporte Delta nativo
# ============================================================

# pip install polars

try:
    import polars as pl
    POLARS_OK = True
    print(f"Polars version: {pl.__version__}")
except ImportError:
    POLARS_OK = False
    print("Instala con: pip install polars")

In [ ]:
# ============================================================
# LEER DELTA CON POLARS
# ============================================================

if DELTA_OK and POLARS_OK:
    # Polars lee tablas Delta nativamente
    df_pl = pl.read_delta(TABLE)
    
    print(f"Polars leyo {len(df_pl)} filas\n")
    
    # Analisis con la API de Polars (mas rapida que pandas)
    resultado = (
        df_pl
        .filter(pl.col('estado') == 'completada')
        .group_by('region')
        .agg([
            pl.col('ingreso').sum().alias('ingreso_total'),
            pl.col('ingreso').mean().round(2).alias('ticket_promedio'),
            pl.col('venta_id').count().alias('transacciones'),
        ])
        .sort('ingreso_total', descending=True)
    )
    
    print("Polars — Ingreso por region (solo completadas):")
    print(resultado)
elif not POLARS_OK:
    print("Polars no disponible. Instala con: pip install polars")

---
## 7. Pipeline incremental

In [ ]:
# ============================================================
# PIPELINE QUE CORRE CADA DIA
# ============================================================

# En produccion, este script corre con cron o Airflow
# Cada dia:
#   1. Lee datos nuevos de la fuente
#   2. Los valida
#   3. Hace merge a la tabla Delta (insert nuevos, update existentes)
#   4. Registra el resultado

def pipeline_incremental(table_path, datos_nuevos, campo_clave='venta_id'):
    """
    Pipeline incremental que hace merge de datos nuevos
    a una tabla Delta existente.
    """
    resultado = {
        'timestamp': datetime.now().isoformat(),
        'filas_entrada': len(datos_nuevos),
    }
    
    # 1. Validar
    invalidos = datos_nuevos[datos_nuevos['unidades'] <= 0]
    if len(invalidos) > 0:
        datos_nuevos = datos_nuevos[datos_nuevos['unidades'] > 0].copy()
        resultado['rechazados'] = len(invalidos)
    
    # 2. Merge
    dt = DeltaTable(table_path)
    antes = len(dt.to_pandas())
    
    (
        dt.merge(
            source=datos_nuevos,
            predicate=f's.{campo_clave} = t.{campo_clave}',
            source_alias='s',
            target_alias='t',
        )
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )
    
    dt = DeltaTable(table_path)
    despues = len(dt.to_pandas())
    
    resultado['insertados'] = despues - antes
    resultado['actualizados'] = len(datos_nuevos) - resultado['insertados']
    resultado['version'] = dt.version()
    resultado['filas_tabla'] = despues
    
    return resultado

if DELTA_OK:
    # Simular carga diaria
    datos_dia = pd.DataFrame({
        'venta_id': [1, 2, 213, 214, 215],  # 1 y 2 ya existen
        'fecha': pd.Timestamp('2024-07-03'),
        'producto': ['dashboard', 'reporte', 'api', 'pipeline', 'app_web'],
        'region': ['bogota', 'medellin', 'cali', 'bogota', 'manizales'],
        'unidades': [12, 7, 20, 15, -5],  # -5 es invalido
        'precio': [450.0, 280.0, 620.0, 500.0, 350.0],
        'estado': ['completada', 'completada', 'pendiente', 'completada', 'pendiente'],
        'ingreso': [5400.0, 1960.0, 12400.0, 7500.0, -1750.0],
    })
    
    resultado = pipeline_incremental(TABLE, datos_dia)
    
    print("Resultado del pipeline incremental:\n")
    for k, v in resultado.items():
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# HISTORIAL COMPLETO DE OPERACIONES
# ============================================================

if DELTA_OK:
    dt = DeltaTable(TABLE)
    print(f"Historial de la tabla ({dt.version() + 1} versiones):\n")
    
    for entry in dt.history():
        v = entry.get('version', '?')
        op = entry.get('operation', '?')
        ts = str(entry.get('timestamp', '?'))[:19]
        metrics = entry.get('operationMetrics', {})
        
        detail = ''
        if 'numOutputRows' in metrics:
            detail = f"filas={metrics['numOutputRows']}"
        elif 'numTargetRowsInserted' in metrics:
            detail = (f"insert={metrics.get('numTargetRowsInserted', 0)}, "
                     f"update={metrics.get('numTargetRowsUpdated', 0)}, "
                     f"delete={metrics.get('numTargetRowsDeleted', 0)}")
        
        print(f"  v{v}: {op:10s} | {ts} | {detail}")

In [ ]:
# Limpiar
if os.path.exists('delta_avanzado'):
    shutil.rmtree('delta_avanzado')
    print("Archivos de prueba eliminados.")

---
## Resumen

| Concepto | Lo que importa |
|---|---|
| **Transaction log** | JSON en _delta_log/ que registra cada add/remove de archivos Parquet. Checkpoints cada 10 versiones |
| **Merge condicional** | when_matched_update() con campos especificos, no update_all |
| **Merge con delete** | when_matched_delete() para eliminar filas que matchean |
| **Particionamiento** | partition_by=['region'] crea carpetas. Filtros leen solo la particion relevante |
| **Optimize** | Compacta muchos archivos pequenos en pocos grandes. Mismos datos, menos overhead |
| **Vacuum** | Elimina archivos muertos. Libera espacio pero rompe time travel antiguo |
| **DuckDB** | SQL analitico sobre Delta sin Spark. delta_scan() en el FROM |
| **Polars** | Alternativa a pandas. pl.read_delta() lee nativamente |
| **Pipeline incremental** | Merge diario: insert nuevos + update existentes. Con validacion y log |

### Siguiente paso
En el **Notebook 4.2** construimos pipelines ETL/ELT completos para mover datos entre las zonas del Lakehouse de forma automatizada.